# 01 — Data Loading & Validation

**Goal:** audit all four RetailRocket CSV files individually, then inspect property-file compatibility. No model training.

**Important:** The raw CSV files were not supplied with this notebook. All measurements are produced when you run it locally; no statistics are invented.

## 1. Configure paths and inspect file sizes
Place the four CSV files in `data/raw/` relative to the project root. Run this notebook from the project root or change `DATA_DIR`. File sizes are checked **before** full loading.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import gc

# Works whether Jupyter starts in the project root or notebooks/ folder.
CWD = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [CWD, *CWD.parents] if (p / "data" / "raw").is_dir()), CWD.parent if CWD.name == "notebooks" else CWD)
DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORT_DIR = PROJECT_ROOT / "data" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FILES = ["events.csv", "item_properties_part1.csv", "item_properties_part2.csv", "category_tree.csv"]
print("Project root:", PROJECT_ROOT)
for filename in FILES:
    p = DATA_DIR / filename
    print(f"{filename}: {p.stat().st_size / 1024**2:.2f} MiB" if p.is_file() else f"MISSING: {p}")
assert all((DATA_DIR / f).is_file() for f in FILES), "Put all four CSVs in data/raw/ before continuing."

Project root: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks
events.csv: 89.87 MiB
item_properties_part1.csv: 461.88 MiB
item_properties_part2.csv: 389.99 MiB
category_tree.csv: 0.01 MiB


## 2. Sample schemas first
`nrows=1000` keeps this inspection inexpensive. Types inferred from a sample are **not guaranteed** to hold for the whole file.

In [2]:
samples = {}
for filename in FILES:
    sample = pd.read_csv(DATA_DIR / filename, nrows=1000, low_memory=False)
    samples[filename] = sample
    print(f"\n=== {filename} ===")
    print("columns:", sample.columns.tolist())
    print("sample dtypes:\n", sample.dtypes.to_string())
    print(sample.head(3).to_string(index=False))


=== events.csv ===
columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']
sample dtypes:
 timestamp          int64
visitorid          int64
event             object
itemid             int64
transactionid    float64
    timestamp  visitorid event  itemid  transactionid
1433221332117     257597  view  355908            NaN
1433224214164     992329  view  248676            NaN
1433221999827     111016  view  318965            NaN

=== item_properties_part1.csv ===
columns: ['timestamp', 'itemid', 'property', 'value']
sample dtypes:
 timestamp     int64
itemid        int64
property     object
value        object
    timestamp  itemid   property                           value
1435460400000  460429 categoryid                            1338
1441508400000  206783        888         1116713 960601 n277.200
1439089200000  395014        400 n552.000 639502 n720.000 424566

=== item_properties_part2.csv ===
columns: ['timestamp', 'itemid', 'property', 'value']
sample dtypes:
 t

## 3. Memory-aware full-file audit
Read chunks instead of loading all CSVs simultaneously. Record row counts, missing values, exact duplicate rows, distinct ID values, timestamp ranges, and chunk memory estimates. Exact duplicate detection stores row hashes (rather than entire rows), which still consumes memory and has an extremely small collision risk. For a large production audit, use external sorting or a database for collision-free deduplication.

In [3]:
from collections import Counter

def audit_csv(path, chunksize=200_000):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    missing = Counter(); uniques = {c: set() for c in header if c in {"visitorid", "itemid", "categoryid", "parentid", "transactionid", "property", "event"}}
    row_hashes = set(); duplicate_count = 0; rows = 0; peak_chunk_mb = 0.0
    time_min = None; time_max = None; observed_types = {c: set() for c in header}
    for chunk in pd.read_csv(path, chunksize=chunksize, low_memory=False):
        rows += len(chunk)
        peak_chunk_mb = max(peak_chunk_mb, chunk.memory_usage(deep=True).sum()/1024**2)
        missing.update(chunk.isna().sum().to_dict())
        for col in header: observed_types[col].add(str(chunk[col].dtype))
        for col in uniques: uniques[col].update(chunk[col].dropna().unique().tolist())
        hashes = pd.util.hash_pandas_object(chunk, index=False).to_numpy(dtype="uint64")
        for h in hashes:
            h = int(h)
            if h in row_hashes: duplicate_count += 1
            else: row_hashes.add(h)
        if "timestamp" in chunk.columns:
            ts = pd.to_numeric(chunk["timestamp"], errors="coerce").dropna()
            if not ts.empty:
                lo, hi = int(ts.min()), int(ts.max())
                time_min = lo if time_min is None else min(time_min, lo)
                time_max = hi if time_max is None else max(time_max, hi)
        del chunk
    def utc(ms):
        if ms is None: return None
        return pd.to_datetime(ms, unit="ms", utc=True, errors="coerce").isoformat()
    return {"file":path.name, "rows":rows, "columns":header, "observed_chunk_dtypes":{k:sorted(v) for k,v in observed_types.items()},
            "missing":dict(missing), "exact_duplicate_rows_hash_based":duplicate_count,
            "unique_ids":{k:len(v) for k,v in uniques.items()}, "time_start_utc":utc(time_min), "time_end_utc":utc(time_max),
            "peak_chunk_memory_mib":round(peak_chunk_mb,2), "file_size_mib":round(path.stat().st_size/1024**2,2)}

# For very large files, set chunksize lower if memory is constrained.
audits = [audit_csv(DATA_DIR / filename) for filename in FILES]
print(pd.DataFrame([{k:v for k,v in a.items() if k not in ("columns","observed_chunk_dtypes","missing","unique_ids")} for a in audits]).to_string(index=False))

                     file     rows  exact_duplicate_rows_hash_based                   time_start_utc                     time_end_utc  peak_chunk_memory_mib  file_size_mib
               events.csv  2756101                              460 2015-05-03T03:00:04.384000+00:00 2015-09-18T02:59:47.788000+00:00                  17.78          89.87
item_properties_part1.csv 10999999                                0        2015-05-10T03:00:00+00:00        2015-09-13T03:00:00+00:00                  28.88         461.88
item_properties_part2.csv  9275903                                0        2015-05-10T03:00:00+00:00        2015-09-13T03:00:00+00:00                  28.89         389.99
        category_tree.csv     1669                                0                             None                             None                   0.03           0.01


## 4. Inspect missing values, cardinalities and types

In [4]:
for a in audits:
    print(f"\n=== {a['file']} ===")
    print("Columns:", a["columns"])
    print("Dtypes:", a["observed_chunk_dtypes"])
    print("Missing:", a["missing"])
    print("Unique IDs:", a["unique_ids"])


=== events.csv ===
Columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']
Dtypes: {'timestamp': ['int64'], 'visitorid': ['int64'], 'event': ['object'], 'itemid': ['int64'], 'transactionid': ['float64']}
Missing: {'timestamp': 0, 'visitorid': 0, 'event': 0, 'itemid': 0, 'transactionid': 2733644}
Unique IDs: {'visitorid': 1407580, 'event': 3, 'itemid': 235061, 'transactionid': 17672}

=== item_properties_part1.csv ===
Columns: ['timestamp', 'itemid', 'property', 'value']
Dtypes: {'timestamp': ['int64'], 'itemid': ['int64'], 'property': ['object'], 'value': ['object']}
Missing: {'timestamp': 0, 'itemid': 0, 'property': 0, 'value': 0}
Unique IDs: {'itemid': 417053, 'property': 1097}

=== item_properties_part2.csv ===
Columns: ['timestamp', 'itemid', 'property', 'value']
Dtypes: {'timestamp': ['int64'], 'itemid': ['int64'], 'property': ['object'], 'value': ['object']}
Missing: {'timestamp': 0, 'itemid': 0, 'property': 0, 'value': 0}
Unique IDs: {'itemid': 417053, 'property

## 5. Check schema compatibility and logical duplicates across property parts
A repeated `(itemid, property)` is **not automatically an error**: values can change over time. We compare schemas and count overlapping full records separately from repeated logical keys. The full-property check below may require substantial RAM; it reads only the four relevant columns and can be run after the chunk audit.

In [5]:
p1 = samples["item_properties_part1.csv"]
p2 = samples["item_properties_part2.csv"]
assert set(p1.columns) == set(p2.columns), "Property schemas differ: inspect before concatenation"
print("Property columns compatible:", p1.columns.tolist())
print("Sample exact overlap:", len(pd.merge(p1.drop_duplicates(),p2.drop_duplicates(),how="inner")))
print("Note: sample overlap is not a full-file overlap measurement.")

Property columns compatible: ['timestamp', 'itemid', 'property', 'value']
Sample exact overlap: 0
Note: sample overlap is not a full-file overlap measurement.


## 6. Verify category relationships before joining
The category tree describes category-to-parent links, **not** a direct item-to-category join. We must locate and validate a category property in the item-property records, including its timestamp and coverage, before mapping event items to categories.

In [6]:
for filename in ["item_properties_part1.csv", "item_properties_part2.csv"]:
    df = samples[filename]
    if "property" in df.columns:
        print(filename, "sample property names:", df["property"].value_counts().head(15).to_dict())
print("Category tree sample:")
print(samples["category_tree.csv"].head().to_string(index=False))

item_properties_part1.csv sample property names: {'888': 138, '790': 93, 'available': 73, 'categoryid': 43, '6': 37, '227': 26, '159': 26, '283': 26, '776': 25, '917': 25, '202': 21, '678': 19, '364': 18, '663': 17, '764': 15}
item_properties_part2.csv sample property names: {'888': 144, '790': 101, 'available': 59, 'categoryid': 38, '6': 36, '283': 35, '364': 27, '202': 26, '776': 26, '159': 25, '678': 23, '112': 22, '917': 22, '764': 21, '451': 18}
Category tree sample:
 categoryid  parentid
       1016     213.0
        809     169.0
        570       9.0
       1691     885.0
        536    1691.0


## 7. Save audit report and interpret it
The report captures observed counts, types, time ranges and missingness. The full property overlap, item-property temporal coverage, category mapping coverage and conflicts require a subsequent targeted integration audit. Do not interpret absent interactions as negative labels.

In [7]:
report_path = REPORT_DIR / "notebook01_data_audit.json"
with report_path.open("w",encoding="utf-8") as f:
    json.dump({"files":audits,"limitations":["Hash-based exact duplicate count has negligible collision risk", "Property cross-part overlap not yet exhaustively measured", "Category mapping coverage not yet validated", "Peak chunk memory is not total process peak memory"]},f,indent=2,default=str)
print("Saved:", report_path.resolve())

Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\reports\notebook01_data_audit.json


## 8. Data integration plan (provisional)

| Source | Planned role | Validation required |
|---|---|---|
| events | Interaction history, popularity, retrieval targets | Timestamp units, event distribution, user/item IDs |
| properties part 1 | Item attributes, historical content features | Schema, timestamped updates, coverage |
| properties part 2 | Additional item attributes | Compatibility, overlaps, conflicts |
| category tree | Hierarchical category features, diversity | Real item-category links, hierarchy integrity |

**Temporal rule:** at prediction time `t`, use only events and item-property versions known by `t`. Future Notebook 02 will define the split, label horizon and negative sampling.

**Next checkpoint:** Run the notebook and share the audit output. We will interpret real counts and inspect any anomalies before proceeding.